In [ ]:
from pathlib import Path
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.features import geometry_mask, rasterize
from rasterio.transform import rowcol
from rasterio.windows import Window
from skimage.morphology import skeletonize
from tqdm import tqdm

# -----------------------------
# Input Paths
# -----------------------------
"""
img_file = r"./Data/2025/stacked_cropped_2025_norm.tif"
train_shp = r"./Data/vector/trainsamples.shp"
test_shp = r"./Data/vector/val_samples.shp"
"""

folder_path = r'./Data/2025/mosaic_2025.data'; vector_path = folder_path+'/vector'
img_file = folder_path+'/stacked_cropped_2025_norm.tif'
train_shp = vector_path+'/trainsamples.shp'
test_shp = vector_path+'/val_samples.shp'


boundary_shp = folder_path+'/vector/studyarea_MKDelta.shp'
output_tif_normalized = folder_path+'/stacked_cropped_2025_norm.tif'

# -----------------------------
# Patch & Class Configurations
# -----------------------------
patch_size = 24
half_p = patch_size // 2

CLASS_CONFIG = {
    1.0: {"method": "centerline", "stride": 4, "min_frac": 0.20},
    2.0: {"method": "grid", "stride": 6, "min_frac": 0.30},
    3.0: {"method": "grid", "stride": 6, "min_frac": 0.30},
    4.0: {"method": "centerline", "stride": 2, "min_frac": 0.20}, 
    5.0: {"method": "grid", "stride": 4, "min_frac": 0.20},
}
DEFAULT_CONFIG = {"method": "grid", "stride": 6, "min_frac": 0.30}


def extract_patches_from_shp(shp_path, src_raster, is_train=True):
    gdf_raw = gpd.read_file(shp_path)
    if gdf_raw.crs != src_raster.crs:
        gdf_raw = gdf_raw.to_crs(src_raster.crs)

    gdf = gdf_raw.explode(index_parts=False).reset_index(drop=True)

    X_list = []
    y_list = []
    poly_ids = []

    shp_name = Path(shp_path).name
    num_bands = src_raster.count  

    for idx, row in tqdm(
        gdf.iterrows(), total=len(gdf), desc=f"Processing {shp_name}"
    ):
        geom = row.geometry
        if geom is None or geom.is_empty:
            continue

        class_id = int(row["Code"])
        cfg = CLASS_CONFIG.get(row["Code"], DEFAULT_CONFIG)
        method, stride, min_frac = (
            cfg["method"],
            cfg["stride"],
            cfg["min_frac"],
        )

        minx, miny, maxx, maxy = geom.bounds
        r_min, c_min = rowcol(src_raster.transform, minx, maxy)
        r_max, c_max = rowcol(src_raster.transform, maxx, miny)

        r0, r1 = max(0, min(r_min, r_max)), min(
            src_raster.height, max(r_min, r_max)
        )
        c0, c1 = max(0, min(c_min, c_max)), min(
            src_raster.width, max(c_min, c_max)
        )

        if (r1 <= r0) or (c1 <= c0):
            continue

        # window (r_center, c_center)
        centers = []
        if method == "centerline":
            sub_transform = rasterio.windows.transform(
                Window(c0, r0, c1 - c0, r1 - r0), src_raster.transform
            )
            poly_mask = rasterize(
                [(geom, 1)],
                out_shape=(r1 - r0, c1 - c0),
                transform=sub_transform,
                fill=0,
                dtype=np.uint8,
            )
            if np.sum(poly_mask) == 0:
                continue
            skeleton = skeletonize(poly_mask > 0)
            coords = np.argwhere(skeleton)
            if len(coords) == 0:
                coords = np.argwhere(poly_mask > 0)
            for r_loc, c_loc in coords[::stride]:
                centers.append((r0 + r_loc, c0 + c_loc))
        else:
            for r in range(r0, r1 - patch_size + 1, stride):
                for c in range(c0, c1 - patch_size + 1, stride):
                    centers.append((r + half_p, c + half_p))

        # Cắt patch thực tế từ Raster
        for r_center, c_center in centers:
            r_start = max(0, r_center - half_p)
            r_end = min(src_raster.height, r_center + half_p)
            c_start = max(0, c_center - half_p)
            c_end = min(src_raster.width, c_center + half_p)

            # Omit not-full-size patches in boundaries 
            if (
                (r_end - r_start) != patch_size
                or (c_end - c_start) != patch_size
            ):
                continue

            win = Window(c_start, r_start, patch_size, patch_size)

            # Check phủ min_frac
            wt = rasterio.windows.transform(win, src_raster.transform)
            mask = geometry_mask(
                [geom],
                transform=wt,
                invert=False,
                out_shape=(patch_size, patch_size),
            )
            frac = np.count_nonzero(~mask) / (patch_size * patch_size)

            if frac >= min_frac:
                patch_data = src_raster.read(
                    window=win
                )  # output Shape: (Bands, H, W)

                # shape 3D
                if patch_data.shape == (num_bands, patch_size, patch_size):
                    # Error NaN/Inf
                    if np.isnan(patch_data).any() or np.isinf(patch_data).any():
                        continue

                    X_list.append(patch_data)
                    y_list.append(class_id)
                    poly_ids.append(idx)

    if len(X_list) == 0:
        return (
            np.empty((0, patch_size, patch_size, num_bands)),
            np.empty((0,)),
            np.empty((0,)),
        )

    # Convert from (N, C, H, W) to (N, H, W, C) for Keras
    X_arr = np.array(X_list)  # (N, C, H, W)
    X_arr = np.transpose(X_arr, (0, 2, 3, 1))  # Result: (N, H, W, C)

    return X_arr, np.array(y_list), np.array(poly_ids)


# -----------------------------
# Extraction
# -----------------------------
with rasterio.open(img_file) as src:
    print(
        f"--- Read file img: {Path(img_file).name} ({src.count} band) ---"
    )

    print("\n--- Extracted TRAIN ---")
    X_train, y_train, poly_ids_train = extract_patches_from_shp(
        train_shp, src, is_train=True
    )

    print("\n--- Extracted VALIDATION/TEST ---")
    X_test, y_test, _ = extract_patches_from_shp(test_shp, src, is_train=False)

print(
    f"\n[Kết quả Train] Shape X_train: {X_train.shape} | Shape y_train: {y_train.shape}"
)
print(
    f"[Kết quả Test]  Shape X_test:  {X_test.shape} | Shape y_test:  {y_test.shape}"
)
print(f"-> No of band for Keras Model: {X_train.shape[-1]}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 1. convert patch 3D (24, 24, 12) to Feature Vector 1D (12 bands)
# Avarage values of pixel (Height=axis 1, Width=axis 2) for each band (Channels=axis 3)
X_train_flat = X_train.mean(axis=(1, 2))  # Shape : (6047, 12)
X_test_flat = X_test.mean(axis=(1, 2))    # Shape : (2500, 12)

print(f"Shape X_train_flat average: {X_train_flat.shape}")

# 2. Train Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_flat, y_train)

# 3. Extract Band Importance
importances = rf.feature_importances_

band_mapping = [
    'C22',
    'Sigma0_HH',
    'Sigma0_HH_Contrast',
    'Sigma0_HH_GLCMCorrelation',
    'Sigma0_HH_GLCMMean',
    'Sigma0_HH_GLCMVariance',
    'Sigma0_HH_Homogeneity',
    'Sigma0_HV',
    'Sigma0_HV_GLCMCorrelation',
    'Sigma0_HV_GLCMMean',
    'Sigma0_HV_GLCMVariance',
    'NDVI'
]

# Dataframe 12 Band
importance_df = pd.DataFrame({
    'Band': band_mapping,
    'Importance': importances
}).sort_values(by='Importance', ascending=True) 

print("\n=== BAND IMPORTANCE ===")
print(importance_df.sort_values(by='Importance', ascending=False))

# 4. Figure 
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11

fig, ax = plt.subplots(figsize=(8, 6), dpi=300)

bars = ax.barh(importance_df['Band'], importance_df['Importance'], 
               color='#2b5c8f', edgecolor='black', linewidth=0.8, height=0.65)

# bar values
for bar in bars:
    width = bar.get_width()
    ax.text(width + 0.002, bar.get_y() + bar.get_height()/2, 
            f'{width:.3f}', ha='left', va='center', fontsize=9, color='#333333')

ax.set_xlabel('Gini Importance (Mean Decrease in Impurity)', fontsize=11, fontweight='bold', labelpad=10)
ax.set_ylabel('Features / Spectral Bands', fontsize=11, fontweight='bold', labelpad=10)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.0)
ax.spines['bottom'].set_linewidth(1.0)

ax.xaxis.grid(True, linestyle='--', alpha=0.5, color='gray')
ax.set_axisbelow(True)

ax.set_xlim(0, importance_df['Importance'].max() * 1.18)

plt.tight_layout()

# Xuất file
plt.savefig('Feature_Importance_RF.png', dpi=300, bbox_inches='tight')
plt.savefig('Feature_Importance_RF.pdf', bbox_inches='tight')

plt.show()

In [ ]:
# Choose selected bands 
#  band Importance >= 0.104
selected_band_indices = np.where(importances >= 0.112)[0]
num_bands = X_train.shape[-1]

print(f"Initial band: {num_bands}")
print(f"retained band: {len(selected_band_indices)}")
print(f"selected band: {selected_band_indices}")

# Filtering X bands
X_train_selected = X_train[:, :, :, selected_band_indices]
X_test_selected  = X_test[:, :, :, selected_band_indices]

print(f"\nShape X_train new for CNN: {X_train_selected.shape}")
print(f"Shape X_test new for CNN:   {X_test_selected.shape}")

# Saved (.npz) for PyTorch / TensorFlow
npz_path = r'./Data/npz'
np.savez_compressed(
    npz_path+'/dataset2025_ready2.npz',
    X_train=X_train_selected,
    y_train=y_train,
    X_test=X_test_selected,
    y_test=y_test,
    poly_ids_train=poly_ids_train, 
)
print("\n--> Saved 'dataset2025_ready2.npz'.")